# What is a top slot worth? Experiments on Expedia hotel search

Expedia's hotel search logs from November 2012 to June 2013: 9.9M hotel impressions from 400k searches. In about 30% of searches the hotels were shown in random order, which turns the file into a randomised experiment on position.

This notebook runs the whole workflow:

1. checks before trusting anything
2. the A/B test: slots 1-4 vs slots 6-10, clustered standard errors, segments, Benjamini-Hochberg
3. cross-checks: same-hotel comparison and CUPED
4. power and sample size, within-search and between-search
5. A/A tests and peeking
6. a geo-lift test design with synthetic control

Analysis is SQL (DuckDB). Python handles the simulations and the synthetic control. Same code as the [GitHub repo](https://github.com/OmidAkrami/hotel-search-experiments).

In [ ]:
import glob, os, pathlib, shutil, subprocess
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

ON_KAGGLE = os.path.exists("/kaggle/input")
# WORK holds results/, RUN holds the parquet and the database (see Data below)
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path("..").resolve()
RUN = pathlib.Path("/kaggle/temp") if ON_KAGGLE else WORK
(WORK / "results").mkdir(exist_ok=True)
(RUN / "data").mkdir(parents=True, exist_ok=True)

BLUE, RED, GREY = "#2f5d8a", "#c0392b", "#9aa5b1"
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12})

## Data

On Kaggle the competition files sit under `/kaggle/input`. `train.csv` (2.4 GB of text with the string `NULL` for missing values) gets converted once to typed parquet, about 150 MB. The parquet and the DuckDB database hold row-level competition data, so on Kaggle they go to `/kaggle/temp`, which isn't saved with the notebook. Only the aggregated CSVs in `results/` end up in the notebook's Output.

In [ ]:
# column types spelled out so DuckDB doesn't guess from the first rows
cols = {
    "srch_id": "INTEGER", "date_time": "TIMESTAMP", "site_id": "SMALLINT",
    "visitor_location_country_id": "SMALLINT", "visitor_hist_starrating": "DOUBLE",
    "visitor_hist_adr_usd": "DOUBLE", "prop_country_id": "SMALLINT", "prop_id": "INTEGER",
    "prop_starrating": "TINYINT", "prop_review_score": "DOUBLE", "prop_brand_bool": "TINYINT",
    "prop_location_score1": "DOUBLE", "prop_location_score2": "DOUBLE",
    "prop_log_historical_price": "DOUBLE", "position": "SMALLINT", "price_usd": "DOUBLE",
    "promotion_flag": "TINYINT", "srch_destination_id": "INTEGER", "srch_length_of_stay": "SMALLINT",
    "srch_booking_window": "SMALLINT", "srch_adults_count": "TINYINT", "srch_children_count": "TINYINT",
    "srch_room_count": "TINYINT", "srch_saturday_night_bool": "TINYINT",
    "srch_query_affinity_score": "DOUBLE", "orig_destination_distance": "DOUBLE", "random_bool": "TINYINT",
}
for i in range(1, 9):
    cols[f"comp{i}_rate"] = "TINYINT"
    cols[f"comp{i}_inv"] = "TINYINT"
    cols[f"comp{i}_rate_percent_diff"] = "DOUBLE"
cols.update({"click_bool": "TINYINT", "gross_bookings_usd": "DOUBLE", "booking_bool": "TINYINT"})

parquet = RUN / "data" / "train.parquet"
if not parquet.exists():
    found = (glob.glob("/kaggle/input/**/train.csv", recursive=True)
             or glob.glob("/kaggle/input/**/data.zip", recursive=True)
             or glob.glob(str(RUN / "data" / "data.zip")))
    src = pathlib.Path(found[0])
    if src.suffix == ".zip":
        # data.zip uses Deflate64, which Python's zipfile can't read
        assert shutil.which("unzip"), "needs the unzip command"
        subprocess.run(["unzip", "-q", "-o", str(src), "train.csv", "-d", str(RUN / "data")], check=True)
        src = RUN / "data" / "train.csv"

    spec = "{" + ", ".join(f"'{k}': '{v}'" for k, v in cols.items()) + "}"
    tmp = duckdb.connect()
    tmp.execute("set preserve_insertion_order = false")
    tmp.execute(f"""copy (select * from read_csv('{src}', header = true, nullstr = 'NULL', columns = {spec}))
                    to '{parquet}' (format parquet, compression zstd, row_group_size 250000)""")
    tmp.close()
    if src.name == "train.csv" and str(src).startswith(str(RUN)):
        src.unlink()

os.chdir(RUN)
db = duckdb.connect(str(RUN / "experiments.duckdb"))

def run(sql):
    db.execute(sql)

def q(sql):
    return db.sql(sql).df()

q("select count(*) as impressions, count(distinct srch_id) as searches from read_parquet('data/train.parquet')")

## Load and helpers

In [ ]:
run("""
-- 00_load.sql
-- data/train.parquet comes from scripts/prepare_data.py (Kaggle's train.csv, typed).

create or replace view raw_train as
select * from read_parquet('data/train.parquet');


-- Helpers for p-values. Neither DuckDB nor Postgres ships a normal CDF, so
-- this is the Abramowitz & Stegun 7.1.26 approximation of erf (error < 1.5e-7),
-- which is plenty for a p-value.
create or replace macro _as_t(x) as 1 / (1 + 0.3275911 * x);

create or replace macro _erf_pos(x) as
    1 - (  0.254829592 * _as_t(x)
         - 0.284496736 * pow(_as_t(x), 2)
         + 1.421413741 * pow(_as_t(x), 3)
         - 1.453152027 * pow(_as_t(x), 4)
         + 1.061405429 * pow(_as_t(x), 5)) * exp(-x * x);

create or replace macro norm_cdf(z) as
    case when z >= 0 then 0.5 * (1 + _erf_pos(z / sqrt(2)))
         else 0.5 * (1 - _erf_pos(-z / sqrt(2))) end;

create or replace macro p_two_sided(z) as 2 * (1 - norm_cdf(abs(z)));

-- chi-square upper tail via the Wilson-Hilferty cube-root approximation
create or replace macro chisq_p(x, k) as
    1 - norm_cdf((pow(x / k, 1.0 / 3) - (1 - 2.0 / (9 * k))) / sqrt(2.0 / (9 * k)));
""")

In [ ]:
q("select round(norm_cdf(1.96), 4) as cdf_196, round(p_two_sided(1.96), 4) as p_196")

## 1. Staging

One row per hotel shown, one row per search. Slots 5, 11, 17 and 23 are nearly empty in the file and get flagged; the rest of the page falls into blocks between them. Each search also gets two flags for list length (all nine slots 1-4 and 6-10 shown, all thirty slots up to 34 shown), because short lists have a much higher click-through per hotel and would bias any comparison between slots.

In [ ]:
run("""
-- 01_staging.sql
-- Two tables: one row per hotel shown (impressions) and one row per search.
--
-- Each srch_id is one hotel search on an Expedia site.
-- For roughly 30% of searches Expedia showed the hotels in random order
-- (random_bool = 1) instead of its own ranking. That random shuffle is the
-- experiment everything in 03 and 04 is built on.
--
-- Slots 5, 11, 17 and 23 are nearly empty (in the random arm, about 2k
-- impressions each vs 120k for slot 4, see chk_slot_counts in 02). They look
-- reserved for something that isn't in the data, so I flag them and leave
-- them out of the position work.
-- Blocks follow those gaps: 1-4, 6-10, 12-16, 18-22, 24+.
--
-- Search length matters for anything that compares slots. A search with 8
-- hotels has all four top slots but only some of 6-10, and short lists have a
-- much higher click-through per hotel (one click shared by fewer hotels). If
-- every search counts, slots 1-4 get extra weight from short lists and look
-- better than they are. So searches carries two flags:
--   full_top10   all nine slots 1-4 and 6-10 present (used by the A/B test)
--   full_top34   all thirty slots up to 34 present (used by the slot curve)

create or replace table impressions as
select
    srch_id,
    date_time                                               as search_ts,
    cast(date_time as date)                                 as search_date,
    site_id,
    visitor_location_country_id                             as visitor_country,
    srch_destination_id                                     as destination_id,
    prop_id                                                 as hotel_id,
    position,
    position in (5, 11, 17, 23)                             as reserved_slot,
    case
        when position <= 4  then '1-4'
        when position <= 10 then '6-10'
        when position <= 16 then '12-16'
        when position <= 22 then '18-22'
        else '24+'
    end                                                     as slot_block,
    random_bool = 1                                         as is_random,
    click_bool  = 1                                         as clicked,
    booking_bool = 1                                        as booked,
    coalesce(gross_bookings_usd, 0)                         as booking_value,
    price_usd,
    -- 0 stars means unknown or not shown, 0 review score means no reviews
    nullif(prop_starrating, 0)                              as stars,
    nullif(prop_review_score, 0)                            as review_score,
    prop_brand_bool = 1                                     as is_chain,
    promotion_flag = 1                                      as on_promotion,
    prop_location_score1                                    as loc_score1,
    prop_location_score2                                    as loc_score2,
    srch_length_of_stay                                     as nights,
    srch_booking_window                                     as days_ahead,
    srch_adults_count                                       as adults,
    srch_children_count                                     as children,
    -- where the hotel's price sits inside its own search, 0 = cheapest
    percent_rank() over (partition by srch_id order by price_usd) as price_pct_in_search
from raw_train;


create or replace table searches as
select
    srch_id,
    any_value(search_ts)                                    as search_ts,
    any_value(search_date)                                  as search_date,
    any_value(site_id)                                      as site_id,
    any_value(visitor_country)                              as visitor_country,
    any_value(destination_id)                               as destination_id,
    bool_or(is_random)                                      as is_random,
    count(*)                                                as hotels_shown,
    count(*) filter (where clicked)                         as clicks,
    bool_or(booked)                                         as has_booking,
    sum(booking_value)                                      as booking_value,
    min(position) filter (where clicked)                    as first_click_position,
    max(position) filter (where clicked)                    as deepest_click_position,
    min(position) filter (where booked)                     as booked_position,
    any_value(nights)                                       as nights,
    any_value(days_ahead)                                   as days_ahead,
    any_value(adults)                                       as adults,
    any_value(children)                                     as children,
    any_value(dayofweek(search_ts)) in (0, 6)               as weekend_search,
    count(*) filter (where not reserved_slot and position <= 10) = 9  as full_top10,
    count(*) filter (where not reserved_slot and position <= 34) = 30 as full_top34
from impressions
group by srch_id;


-- week number with Monday starts, used by the time-based checks and the geo panel
create or replace macro week_start(d) as cast(date_trunc('week', d) as date);
""")

## 2. Checks before any test

The naive readout (booking rate by group) is the trap here. Then: how was the file sampled, was assignment the same everywhere, and are the hotels in the top slots really the same kind as the ones below?

In [ ]:
run("""
-- 02_checks.sql
-- What has to be true before any test result means anything.
""")

In [ ]:
run("""
-- 1. Size of things
create or replace view chk_overview as
select
    count(*)                                                    as impressions,
    count(distinct srch_id)                                     as searches,
    count(distinct hotel_id)                                    as hotels,
    count(distinct destination_id)                              as destinations,
    count(distinct site_id)                                     as sites,
    min(search_date)                                            as first_day,
    max(search_date)                                            as last_day,
    round(100.0 * count(distinct srch_id) filter (where is_random)
          / count(distinct srch_id), 1)                         as pct_searches_random
from impressions;
""")

In [ ]:
run("""
-- 2. How the file was sampled. Every single search has at least one click,
-- in both groups. Then 94% of ranked searches end in a booking against 13%
-- of random ones. A ranking 7x better than shuffling isn't believable; the
-- likely story is that Expedia sampled the two groups differently and kept
-- ranked searches with a booking far more often. Either way, booking rates
-- between the groups say nothing about the ranking. See 05 for what can be
-- compared.
create or replace view chk_sampling as
select
    case when is_random then 'random order' else 'Expedia ranking' end as arm,
    count(*)                                                    as searches,
    round(100.0 * avg(case when clicks > 0 then 1 else 0 end), 1) as pct_with_click,
    round(100.0 * avg(case when has_booking then 1 else 0 end), 1) as pct_with_booking,
    round(avg(clicks), 3)                                       as clicks_per_search,
    round(100.0 * avg(case when clicks = 1 then 1 else 0 end), 1) as pct_exactly_one_click
from searches
group by 1
order by 1;
""")

In [ ]:
run("""
-- 3. Assignment to random order was not the same everywhere. Among sites with
-- 1,000+ searches the random share goes from 0% to 51% (one tiny site goes
-- higher), and the chi-square test of "same share on every site" is off the
-- scale. So:
--   * comparing the two arms needs to be done within site (05 does that)
--   * inside the random arm nothing changes: positions are shuffled within
--     each search, so site mix can't line up with position
create or replace view chk_assignment_by_site as
with s as (
    select site_id, count(*) as searches, count(*) filter (where is_random) as random_searches
    from searches
    group by site_id
),
tot as (
    select sum(random_searches) * 1.0 / sum(searches) as p from s
)
select
    s.site_id,
    s.searches,
    s.random_searches,
    round(100.0 * s.random_searches / s.searches, 1)            as pct_random,
    round((s.random_searches - s.searches * t.p) / sqrt(s.searches * t.p * (1 - t.p)), 1) as z_vs_overall
from s
cross join tot t
order by s.searches desc;


-- The chi-square approximation needs an expected count of 5+ in every cell,
-- so sites too small for that are left out (sites_used shows how many stay).
create or replace view chk_assignment_test as
with s as (
    select site_id, count(*) as n, count(*) filter (where is_random) as r
    from searches
    group by site_id
),
p_all as (select sum(r) * 1.0 / sum(n) as p from s),
kept as (
    select s.*
    from s cross join p_all a
    where s.n * a.p >= 5 and s.n * (1 - a.p) >= 5
),
tot as (select sum(r) * 1.0 / sum(n) as p, count(*) as k from kept),
cells as (
    select
        pow(s.r - s.n * t.p, 2) / (s.n * t.p)
      + pow((s.n - s.r) - s.n * (1 - t.p), 2) / (s.n * (1 - t.p)) as contrib
    from kept s cross join tot t
)
select
    'same random share on every site'                           as hypothesis,
    (select count(*) from s)                                    as sites,
    (select k from tot)                                         as sites_used,
    (select k from tot) - 1                                     as dof,
    round(sum(contrib), 1)                                      as chi_square,
    chisq_p(sum(contrib), (select k from tot) - 1)              as p_value
from cells;
""")

In [ ]:
run("""
-- 4. Over time the share stays between 27% and 33% a week. That's more
-- wobble than pure chance at ~11k searches a week, which fits the site mix
-- shifting from week to week (shares differ by site, see 3). No sign of the
-- test being switched on or off midway.
create or replace view chk_assignment_by_week as
select
    week_start(search_date)                                     as week,
    count(*)                                                    as searches,
    round(100.0 * avg(case when is_random then 1 else 0 end), 1) as pct_random
from searches
group by 1
order by 1;
""")

In [ ]:
run("""
-- 5. Impressions per slot in the random arm. 5, 11, 17 and 23 are nearly empty.
create or replace view chk_slot_counts as
select
    position,
    count(*)                                                    as impressions,
    reserved_slot
from impressions
where is_random
group by position, reserved_slot
order by position;
""")

In [ ]:
run("""
-- 6. Randomisation check inside the random arm: are the hotels in slots 1-4
-- the same kind as the hotels in slots 6-10? If the shuffle is random they
-- should be, give or take noise. Standardised mean difference (SMD): the gap
-- in means divided by the pooled standard deviation. Under 0.1 is the usual
-- "balanced" bar. The ranked arm is there for contrast.
--
-- Result: stars, price, review score, chain, promotion and location score 1
-- are balanced. Location score 2 is not: top slots get hotels with slightly
-- better scores and far fewer missing ones. The share missing climbs steadily
-- from 11% at slot 1 to 24% at slot 14 (8% to 17% among searches that show
-- all 34 slots, so list length isn't behind it; see chk_loc2_by_slot). I
-- read that as a shuffle that wasn't fully uniform, more than an effect of
-- the "at least one click" sampling: location score 2 barely predicts clicks
-- (correlation 0.06, chk_loc2_click_corr). 04 checks the main result with a
-- same-hotel comparison and a regression adjustment.
create or replace view chk_balance_top_vs_next as
with b as (
    select
        is_random,
        slot_block = '1-4'                                      as top,
        stars, review_score, price_pct_in_search, loc_score1, loc_score2,
        case when loc_score2 is null then 1.0 else 0 end        as loc2_missing,
        case when is_chain then 1.0 else 0 end                  as chain,
        case when on_promotion then 1.0 else 0 end              as promo
    from impressions
    where slot_block in ('1-4', '6-10') and not reserved_slot
      -- same searches as the test in 04
      and srch_id in (select srch_id from searches where full_top10)
),
long as (
    select is_random, top, 'stars' as covariate, stars as v from b
    union all select is_random, top, 'review score', review_score from b
    union all select is_random, top, 'price percentile in search', price_pct_in_search from b
    union all select is_random, top, 'location score 1', loc_score1 from b
    union all select is_random, top, 'location score 2', loc_score2 from b
    union all select is_random, top, 'location score 2 missing', loc2_missing from b
    union all select is_random, top, 'chain hotel', chain from b
    union all select is_random, top, 'on promotion', promo from b
),
m as (
    select
        is_random, covariate,
        avg(v) filter (where top)                               as mean_top,
        avg(v) filter (where not top)                           as mean_next,
        var_samp(v) filter (where top)                          as var_top,
        var_samp(v) filter (where not top)                      as var_next
    from long
    group by is_random, covariate
)
select
    case when is_random then 'random order' else 'Expedia ranking' end as arm,
    covariate,
    round(mean_top, 4)                                          as slots_1_4,
    round(mean_next, 4)                                         as slots_6_10,
    round((mean_top - mean_next) / sqrt((var_top + var_next) / 2), 3) as smd,
    abs((mean_top - mean_next) / sqrt((var_top + var_next) / 2)) < 0.1 as balanced
from m
order by arm desc, abs((mean_top - mean_next) / sqrt((var_top + var_next) / 2)) desc;


-- Location score 2 missing share by slot in the random arm: all searches,
-- and only searches that show all 34 slots (so list length can't explain it)
create or replace view chk_loc2_by_slot as
select
    position,
    count(*)                                                    as impressions,
    round(100 * avg(case when loc_score2 is null then 1.0 else 0 end), 1) as pct_missing_all_searches,
    round(100 * avg(case when loc_score2 is null then 1.0 else 0 end)
          filter (where srch_id in (select srch_id from searches where full_top34)), 1) as pct_missing_full_top34
from impressions
where is_random and not reserved_slot and position <= 34
group by position
order by position;


-- How much location score 2 says about a click, in the test population of 04
create or replace view chk_loc2_click_corr as
select
    count(loc_score2)                                           as impressions_with_score,
    round(corr(case when clicked then 1.0 else 0 end, loc_score2), 3) as corr_click_loc_score2
from impressions
where is_random and not reserved_slot and slot_block in ('1-4', '6-10')
  and srch_id in (select srch_id from searches where full_top10);
""")

In [ ]:
run("""
-- 7. Missing values in the fields used later
create or replace view chk_fill_rates as
select
    round(100.0 * count(stars) / count(*), 1)                   as pct_stars,
    round(100.0 * count(review_score) / count(*), 1)            as pct_review_score,
    round(100.0 * count(loc_score2) / count(*), 1)              as pct_loc_score2,
    round(100.0 * count(price_usd) / count(*), 1)               as pct_price,
    round(100.0 * count(*) filter (where booked and booking_value > 0)
          / nullif(count(*) filter (where booked), 0), 2)        as pct_bookings_with_value
from impressions;
""")

In [ ]:
q("select * from chk_overview")

In [ ]:
q("select * from chk_sampling")

In [ ]:
q("select * from chk_assignment_test")

In [ ]:
q("select * from chk_assignment_by_site limit 15")

In [ ]:
d = q("select * from chk_assignment_by_site where searches >= 1000 order by pct_random")
fig, ax = plt.subplots()
ax.barh(d.site_id.astype(str), d.pct_random, color=[RED if abs(z) > 3 else BLUE for z in d.z_vs_overall])
ax.axvline(30.4, color=GREY, ls="--", lw=1)
ax.set_xlabel("% of searches in random order"); ax.set_ylabel("site")
ax.set_title("Share of random-order searches differs by site (red: |z| > 3)")
plt.tight_layout(); plt.show()

In [ ]:
q("select * from chk_balance_top_vs_next")

In [ ]:
q("select * from chk_loc2_by_slot")

In [ ]:
q("select * from chk_slot_counts where position <= 25")

## 3. Position on its own

In the random arm the hotel in slot 1 is a random one, so slot 1's extra clicks are caused by the slot. In Expedia's ranking the slot and the hotel are mixed up. Only searches that show all slots to 34 are used, so every slot is measured on the same searches.

In [ ]:
run("""
-- 03_position_effect.sql
-- How much does position on the page matter, on its own?
--
-- In Expedia's own ranking, slot 1 gets clicked a lot partly because it's at
-- the top and partly because the algorithm put a good hotel there. Ranked
-- results can't separate the two. In the random arm the hotel in slot 1
-- is a random one, so whatever extra clicks slot 1 gets there are caused by
-- the slot itself.
--
-- One impression per search per slot, so impressions at a given slot are
-- independent across searches and the plain binomial standard error is right
-- for a single slot. (Comparing blocks of slots needs more care, see 04.)
--
-- Only searches that show all 30 slots up to 34 are used (full_top34, see
-- 01). Without that, the top slots carry every short list and the bottom
-- slots only long ones, and the curve comes out steeper than it is.
-- About 55k random and 130k ranked searches qualify (pos_curve impressions
-- at any one slot).
""")

In [ ]:
run("""
-- 1. Click-through, booking and conversion-after-click by slot, both arms
create or replace view pos_curve as
with p as (
    select
        case when is_random then 'random' else 'ranked' end     as arm,
        position,
        count(*)                                                as impressions,
        count(*) filter (where clicked)                         as clicks,
        count(*) filter (where booked)                          as bookings
    from impressions
    where srch_id in (select srch_id from searches where full_top34)
      and not reserved_slot and position <= 34
    group by 1, 2
    having count(*) >= 1000
),
e as (
    select
        *,
        clicks * 1.0 / impressions                              as ctr,
        1.96 * sqrt((clicks * 1.0 / impressions) * (1 - clicks * 1.0 / impressions) / impressions) as half_width
    from p
)
select
    arm,
    position,
    impressions,
    clicks,
    bookings,
    round(100 * ctr, 2)                                         as ctr_pct,
    round(100 * half_width, 2)                                  as ctr_ci95_pts,
    round(100 * (ctr - half_width), 2)                          as ctr_lo,
    round(100 * (ctr + half_width), 2)                          as ctr_hi,
    round(100.0 * bookings / impressions, 3)                    as bookings_per_100_impressions,
    round(100.0 * bookings / nullif(clicks, 0), 1)              as booked_after_click_pct
from e
order by arm, position;
""")

In [ ]:
run("""
-- 2. Index: each slot's click-through relative to slot 1 in the same arm.
-- Only the random column is causal. The ranked column mixes position with
-- the ranking putting better hotels on top, and its level is also inflated by
-- the sampling (02), so the two arms aren't compared slot for slot here.
create or replace view pos_index as
with r as (
    -- unrounded click-through, so the index isn't built on rounded numbers
    select
        position,
        max(clicks * 1.0 / impressions) filter (where arm = 'random') as ctr_random,
        max(clicks * 1.0 / impressions) filter (where arm = 'ranked') as ctr_ranked
    from pos_curve
    group by position
)
select
    position,
    round(100 * ctr_random, 2)                                  as random_ctr,
    round(100 * ctr_ranked, 2)                                  as ranked_ctr,
    round(100 * ctr_random / first_value(ctr_random) over (order by position), 1) as random_index,
    round(100 * ctr_ranked / first_value(ctr_ranked) over (order by position), 1) as ranked_index
from r
where ctr_random is not null and ctr_ranked is not null
order by position;
""")

In [ ]:
run("""
-- 3. Same thing by block of slots (the last block is 24-34 here)
create or replace view pos_blocks as
select
    case when is_random then 'random' else 'ranked' end         as arm,
    case when slot_block = '24+' then '24-34' else slot_block end as slot_block,
    count(*)                                                    as impressions,
    round(100.0 * avg(case when clicked then 1 else 0 end), 2)  as ctr_pct,
    round(100.0 * count(*) filter (where booked) / nullif(count(*) filter (where clicked), 0), 1) as booked_after_click_pct,
    -- share of all clicks in that arm that land in this block
    round(100.0 * count(*) filter (where clicked)
          / sum(count(*) filter (where clicked)) over (partition by is_random), 1) as pct_of_arm_clicks
from impressions
where srch_id in (select srch_id from searches where full_top34)
  and not reserved_slot and position <= 34
group by is_random, slot_block
order by arm, min(position);
""")

In [ ]:
run("""
-- 4. Does position change what happens after the click? In the random arm,
-- compare the share of clicks that turn into a booking in slots 1-4 against
-- each lower block. Both blocks come from the same searches, so the error
-- uses the delta method with the search as the cluster, as in 04.
create or replace view pos_after_click as
with per as (
    -- one row per search and block (every full_top34 search has all blocks)
    select
        srch_id,
        case when slot_block = '24+' then '24-34' else slot_block end as slot_block,
        min(position)                                           as first_slot,
        count(*) filter (where clicked)                         as c,
        count(*) filter (where booked)                          as b
    from impressions
    where is_random and not reserved_slot and position <= 34
      and srch_id in (select srch_id from searches where full_top34)
    group by 1, 2
),
tot as (
    select
        slot_block,
        min(first_slot)                                         as first_slot,
        sum(c)                                                  as clicks,
        sum(b)                                                  as bookings,
        sum(b) * 1.0 / sum(c)                                   as rate
    from per
    group by slot_block
),
-- each block's booking rate linearised per search (delta method)
psi as (
    select p.srch_id, p.slot_block, (p.b - t.rate * p.c) / t.clicks as psi
    from per p
    join tot t on t.slot_block = p.slot_block
),
se as (
    select x.slot_block, sqrt(sum(pow(top.psi - x.psi, 2)))    as se_gap
    from psi x
    join psi top on top.srch_id = x.srch_id and top.slot_block = '1-4'
    group by x.slot_block
)
select
    t.slot_block,
    t.clicks,
    round(100 * t.rate, 2)                                      as booked_after_click_pct,
    round(100 * (top.rate - t.rate), 2)                         as gap_vs_top_pts,
    case when t.slot_block <> '1-4'
         then round(p_two_sided((top.rate - t.rate) / se.se_gap), 4) end as p_value
from tot t
cross join (select rate from tot where slot_block = '1-4') top
join se on se.slot_block = t.slot_block
order by t.first_slot;
""")

In [ ]:
d = q("select * from pos_curve where position <= 36")
fig, ax = plt.subplots()
for arm, c in [("random", BLUE), ("ranked", GREY)]:
    x = d[d.arm == arm]
    ax.errorbar(x.position, x.ctr_pct, yerr=x.ctr_ci95_pts, color=c, marker="o", ms=3, lw=1.5, capsize=2,
                label="random order (position effect)" if arm == "random" else "Expedia ranking (position + relevance)")
ax.set_xlabel("slot"); ax.set_ylabel("click-through, %")
ax.set_title("Slot 1 gets about 3x the clicks of slot 10 even when hotels are shuffled")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
q("select * from pos_index where position <= 16")

In [ ]:
q("select * from pos_blocks")

In [ ]:
q("select * from pos_after_click")

## 4. The A/B test: slots 1-4 vs slots 6-10

Unit: a hotel impression in a random-order search that shows all nine slots 1-4 and 6-10. Standard errors by the delta method with the search as the cluster (the hotels in a search share one user). One long table of (test, metric, segment, search, arm, numerator, denominator) and one query that does the maths for all of it.

In [ ]:
run("""
-- 04_ab_test.sql
-- The A/B test: does showing a hotel in slots 1-4 instead of slots 6-10 change
-- what users do with it?
--
--   population   random-order searches that show all nine slots 1-4 and 6-10
--   unit         a hotel impression
--   treatment    shown in slots 1-4
--   control      shown in slots 6-10
--   primary      click-through rate
--   secondary    bookings per impression, booking value per impression,
--                share of clicks that end in a booking
--
-- The full-list filter matters. A search with 7 hotels has four in slots 1-4
-- and only three in 6-10 (slot 5 is reserved), and short lists have a high
-- click-through per hotel. Counting every search gives slots 1-4 more of
-- those easy impressions and adds about 7 points of lift (83.2% vs 76.6%,
-- see ab_list_length) that has nothing to do with position.
--
-- Impressions aren't independent: the ~25 hotels in one search share one
-- user, who clicks about once. So the error is computed with the search as
-- the unit. Every metric is a ratio (sum of x / sum of y), and the standard
-- error comes from the delta method with searches as clusters (Deng, Knoblich
-- & Lu 2018, "Applying the Delta Method in Metric Analytics"):
--
--   r = sum(num) / sum(den) per arm, n = searches in the test
--   per search: u = (num_T - r_T * den_T) / mean_den_T - (num_C - r_C * den_C) / mean_den_C
--   se(r_T - r_C) = sqrt(sum(u^2)) / n
--
-- Treatment and control impressions come from the same searches, and taking
-- the difference per search before squaring accounts for that covariance.
-- Same idea on the log scale for the relative lift.
--
-- Everything goes into one long table (ab_units) so a single query does the
-- maths for every test, metric and segment.


-- covariate for the CUPED run (section 4): how click-prone is a hotel of
-- this kind, measured only on searches before 1 Feb 2013 so it can't be
-- affected by anything later. "Kind" = stars x review band x price third
-- x location score 2 band x chain. Expressed as observed / expected clicks,
-- where expected uses the slot click-through, so slot luck doesn't leak in.
create or replace table cuped_cells as
with pre as (
    select *
    from impressions
    where is_random and not reserved_slot and search_date < date '2013-02-01'
),
slot_ctr as (
    select position, avg(case when clicked then 1.0 else 0 end) as ctr
    from pre
    group by position
),
banded as (
    select
        p.*,
        s.ctr                                                   as expected,
        coalesce(cast(p.stars as varchar), '?')                 as star_band,
        case when p.review_score is null then 'none'
             when p.review_score < 3.5 then 'low'
             when p.review_score < 4.5 then 'mid' else 'high' end as review_band,
        case when p.price_pct_in_search < 1.0 / 3 then 'cheap'
             when p.price_pct_in_search < 2.0 / 3 then 'mid' else 'pricey' end as price_band,
        case when p.loc_score2 is null then 'missing'
             else cast(least(floor(p.loc_score2 / 0.05), 6) as varchar) end as loc_band
    from pre p
    join slot_ctr s on s.position = p.position
)
select
    star_band, review_band, price_band, loc_band, is_chain,
    count(*)                                                    as impressions,
    sum(case when clicked then 1.0 else 0 end) / sum(expected)  as attractiveness
from banded
group by all
having count(*) >= 200;


-- The CUPED population: the test's impressions from Feb-Jun searches, each
-- with its covariate x (1.0 for hotel kinds with no pre-period cell).
create or replace table cuped_units as
with t as (
    select
        i.srch_id,
        case when i.slot_block = '1-4' then 'T' else 'C' end    as arm,
        case when i.clicked then 1.0 else 0 end                 as click,
        coalesce(cast(i.stars as varchar), '?')                 as star_band,
        case when i.review_score is null then 'none'
             when i.review_score < 3.5 then 'low'
             when i.review_score < 4.5 then 'mid' else 'high' end as review_band,
        case when i.price_pct_in_search < 1.0 / 3 then 'cheap'
             when i.price_pct_in_search < 2.0 / 3 then 'mid' else 'pricey' end as price_band,
        case when i.loc_score2 is null then 'missing'
             else cast(least(floor(i.loc_score2 / 0.05), 6) as varchar) end as loc_band,
        i.is_chain
    from impressions i
    where i.is_random and not i.reserved_slot and i.slot_block in ('1-4', '6-10')
      and i.search_date >= date '2013-02-01'
      and i.srch_id in (select srch_id from searches where full_top10)
)
select t.srch_id, t.arm, t.click, coalesce(c.attractiveness, 1.0) as x
from t
left join cuped_cells c
  on c.star_band = t.star_band and c.review_band = t.review_band
 and c.price_band = t.price_band and c.loc_band = t.loc_band and c.is_chain = t.is_chain;


-- theta is the slope of click on x within arms: the coefficient a regression
-- of click on arm and x gives. Pooling the arms instead would let the gap
-- between them leak into theta, because x isn't balanced between the arms
-- here (the location score 2 imbalance, see 02).
create or replace view ab_cuped_covariate as
with m as (
    select
        *,
        avg(click) over (partition by arm)                      as click_arm_mean,
        avg(x) over (partition by arm)                          as x_arm_mean
    from cuped_units
)
select
    count(*)                                                    as impressions,
    corr(click, x)                                              as corr_click_x,
    sum((click - click_arm_mean) * (x - x_arm_mean))
        / sum(pow(x - x_arm_mean, 2))                           as theta,
    avg(x)                                                      as x_mean,
    avg(x) filter (where arm = 'T')                             as x_mean_slots_1_4,
    avg(x) filter (where arm = 'C')                             as x_mean_slots_6_10
from m;


create or replace table ab_units as
with base as (
    select
        i.*,
        case when i.slot_block = '1-4' then 'T' when i.slot_block = '6-10' then 'C' end as arm,
        case when i.clicked then 1.0 else 0 end                 as click,
        case when i.booked then 1.0 else 0 end                  as booking,
        coalesce(cast(i.stars as varchar), '?')                 as star_band,
        case when i.review_score is null then 'none'
             when i.review_score < 3.5 then 'low'
             when i.review_score < 4.5 then 'mid' else 'high' end as review_band,
        case when i.price_pct_in_search < 1.0 / 3 then 'cheap'
             when i.price_pct_in_search < 2.0 / 3 then 'mid' else 'pricey' end as price_band,
        case when i.loc_score2 is null then 'missing'
             else cast(least(floor(i.loc_score2 / 0.05), 6) as varchar) end as loc_band
    from impressions i
    where i.is_random and not i.reserved_slot
),
top as (
    select *
    from base
    where arm is not null
      and srch_id in (select srch_id from searches where full_top10)
),
seg as (
    -- one row per impression per segment dimension it belongs to
    select srch_id, arm, click, 'hotel stars' as dim,
           case when stars is null then 'unknown' when stars <= 2 then '1-2 stars'
                when stars = 3 then '3 stars' else '4-5 stars' end as segment from top
    union all
    select srch_id, arm, click, 'review score',
           case when review_score is null then 'no reviews' when review_score < 3.5 then 'under 3.5'
                when review_score < 4.5 then '3.5 to 4' else '4.5 to 5' end from top
    union all
    select srch_id, arm, click, 'price within search',
           case when price_band = 'cheap' then 'cheapest third' when price_band = 'mid' then 'middle third'
                else 'priciest third' end from top
    union all
    select srch_id, arm, click, 'booking window',
           case when days_ahead <= 3 then '0-3 days' when days_ahead <= 30 then '4-30 days'
                else '31+ days' end from top
    union all
    select srch_id, arm, click, 'length of stay',
           case when nights = 1 then '1 night' when nights <= 3 then '2-3 nights' else '4+ nights' end from top
    union all
    select srch_id, arm, click, 'party',
           case when children > 0 then 'with children' else 'adults only' end from top
    union all
    select srch_id, arm, click, 'visitor',
           case when visitor_country = 219 then 'country 219 (largest)' else 'all other countries' end from top
    union all
    select srch_id, arm, click, 'hotel type',
           case when is_chain then 'chain' else 'independent' end from top
    union all
    select srch_id, arm, click, 'promotion',
           case when on_promotion then 'on promotion' else 'no promotion' end from top
),
pairs as (
    -- each non-reserved slot and the next one down, to slot 37 (32 pairs)
    select position as upper_slot, lead(position) over (order by position) as lower_slot
    from (select distinct position from base where position <= 37) d
),
pair_rows as (
    -- only searches that show both slots of the pair, so each pair is
    -- compared inside the same searches
    select
        'slot ' || p.upper_slot || ' vs ' || p.lower_slot        as segment,
        a.srch_id,
        a.click                                                 as click_upper,
        b.click                                                 as click_lower
    from pairs p
    join base a on a.position = p.upper_slot
    join base b on b.srch_id = a.srch_id and b.position = p.lower_slot
    where p.lower_slot is not null
),
adjacent as (
    select segment, srch_id, 'T' as arm, click_upper as click from pair_rows
    union all
    select segment, srch_id, 'C', click_lower from pair_rows
),
cuped as (
    select srch_id, arm, click, x from cuped_units
),
cuped_theta as (
    select theta, x_mean from ab_cuped_covariate
)
-- main test, four metrics
select 'top slots' as test, 'ctr' as metric, 'all' as dim, 'all' as segment,
       srch_id, arm, sum(click) as num, count(*) as den
from top group by all
union all
select 'top slots', 'bookings per impression', 'all', 'all', srch_id, arm, sum(booking), count(*)
from top group by all
union all
select 'top slots', 'booking value per impression', 'all', 'all', srch_id, arm, sum(booking_value), count(*)
from top group by all
union all
select 'top slots', 'booked after click', 'all', 'all', srch_id, arm, sum(booking), sum(click)
from top where clicked group by all
-- segments
union all
select 'top slots', 'ctr', dim, segment, srch_id, arm, sum(click), count(*)
from seg group by all
-- adjacent slots
union all
select 'adjacent slots', 'ctr', 'slot pair', segment, srch_id, arm, sum(click), count(*)
from adjacent group by all
-- CUPED: same test on Feb-Jun searches, raw and adjusted
union all
select 'top slots, Feb-Jun', 'ctr', 'cuped', 'unadjusted', srch_id, arm, sum(click), count(*)
from cuped group by all
union all
select 'top slots, Feb-Jun', 'ctr', 'cuped', 'adjusted', c.srch_id, c.arm,
       sum(c.click - t.theta * (c.x - t.x_mean)), count(*)
from cuped c cross join cuped_theta t group by all;
""")

In [ ]:
run("""
-- 1. The stats engine: one row per test / metric / segment
create or replace view ab_results as
with tot as (
    select
        test, metric, dim, segment,
        count(distinct srch_id)                                 as searches,
        sum(num) filter (where arm = 'T')                       as num_t,
        sum(den) filter (where arm = 'T')                       as den_t,
        sum(num) filter (where arm = 'C')                       as num_c,
        sum(den) filter (where arm = 'C')                       as den_c
    from ab_units
    group by all
),
est as (
    select
        *,
        num_t / den_t                                           as r_t,
        num_c / den_c                                           as r_c,
        den_t / searches                                        as xbar_t,
        den_c / searches                                        as xbar_c
    from tot
),
lin as (
    select
        u.test, u.metric, u.dim, u.segment, u.srch_id,
        sum(case when u.arm = 'T' then (u.num - e.r_t * u.den) / e.xbar_t else 0 end) as u_t,
        sum(case when u.arm = 'C' then (u.num - e.r_c * u.den) / e.xbar_c else 0 end) as u_c
    from ab_units u
    join est e using (test, metric, dim, segment)
    group by all
),
var as (
    select
        l.test, l.metric, l.dim, l.segment,
        sqrt(sum(pow(l.u_t - l.u_c, 2))) / max(e.searches)      as se_diff,
        sqrt(sum(pow(l.u_t / nullif(e.r_t, 0) - l.u_c / nullif(e.r_c, 0), 2))) / max(e.searches) as se_log_ratio
    from lin l
    join est e using (test, metric, dim, segment)
    group by all
)
select
    e.test,
    e.metric,
    e.dim,
    e.segment,
    e.searches,
    cast(e.den_t as bigint)                                     as denom_t,
    cast(e.den_c as bigint)                                     as denom_c,
    e.r_t                                                       as treatment,
    e.r_c                                                       as control,
    e.r_t - e.r_c                                               as diff,
    v.se_diff,
    v.se_log_ratio,
    e.r_t - e.r_c - 1.96 * v.se_diff                            as diff_lo,
    e.r_t - e.r_c + 1.96 * v.se_diff                            as diff_hi,
    e.r_t / e.r_c - 1                                           as rel_lift,
    exp(ln(e.r_t / e.r_c) - 1.96 * v.se_log_ratio) - 1         as rel_lift_lo,
    exp(ln(e.r_t / e.r_c) + 1.96 * v.se_log_ratio) - 1         as rel_lift_hi,
    (e.r_t - e.r_c) / v.se_diff                                 as z,
    p_two_sided((e.r_t - e.r_c) / v.se_diff)                    as p_value,
    -- what a plain two-proportion test would say (binary metrics only)
    case when e.metric <> 'booking value per impression'
         then sqrt(e.r_t * (1 - e.r_t) / e.den_t + e.r_c * (1 - e.r_c) / e.den_c) end as naive_se,
    case when e.metric <> 'booking value per impression'
         then v.se_diff / sqrt(e.r_t * (1 - e.r_t) / e.den_t + e.r_c * (1 - e.r_c) / e.den_c) end as se_ratio_vs_naive
from est e
join var v using (test, metric, dim, segment)
order by e.test, e.metric, e.dim, e.segment;
""")

In [ ]:
run("""
-- 2. Headline readout, rounded for people
create or replace view ab_headline as
select
    metric,
    searches,
    denom_t                                                     as impressions_t,
    denom_c                                                     as impressions_c,
    round(case when metric = 'booking value per impression' then treatment else 100 * treatment end, 3) as treatment,
    round(case when metric = 'booking value per impression' then control else 100 * control end, 3) as control,
    case when metric = 'booking value per impression' then 'USD' else '%' end as unit,
    round(100 * rel_lift, 1)                                    as lift_pct,
    round(100 * rel_lift_lo, 1)                                 as lift_lo,
    round(100 * rel_lift_hi, 1)                                 as lift_hi,
    round(z, 1)                                                 as z,
    p_value,
    round(se_ratio_vs_naive, 2)                                 as se_vs_naive
from ab_results
where test = 'top slots' and dim = 'all'
order by case metric when 'ctr' then 1 when 'bookings per impression' then 2
                     when 'booking value per impression' then 3 else 4 end;
""")

In [ ]:
run("""
-- 3. Who reacts most to position? Lift of slots 1-4 over 6-10 by segment,
-- on two scales: relative (lift %) and absolute (extra clicks per 100
-- impressions). They answer different questions. A segment with a low base
-- click-through can show a bigger relative lift for the same absolute gain.
-- Each level is tested against the first level of its dimension, on both
-- scales. 16 comparisons per scale, so p-values are adjusted with
-- Benjamini-Hochberg: sort p ascending, adjusted p = min over k >= i of
-- p_k * m / k, done with a window running from the largest p down.
-- Segments of one dimension share searches (a search shows hotels from
-- several segments), so the two estimates in a comparison are correlated.
-- Each estimate is linearised per search (delta method) and the error of the
-- difference comes from the per-search difference of those terms, which
-- carries the covariance, the same way the main test handles 1-4 vs 6-10.
create or replace view ab_segments as
with s as (
    select
        dim, segment, searches, denom_t, denom_c, treatment, control,
        diff, se_diff,
        ln(treatment / control)                                 as log_lift,
        se_log_ratio,
        row_number() over (partition by dim order by segment)   as level_no
    from ab_results
    where test = 'top slots' and metric = 'ctr' and dim not in ('all', 'cuped')
),
pairs as (
    select s.dim, s.segment, r.segment as reference
    from s
    join s r on r.dim = s.dim and r.level_no = 1
    where s.level_no > 1
),
psi as (
    -- per search and segment: the linearised gain (absolute scale) and log
    -- lift (relative scale), slots 1-4 minus slots 6-10
    select
        u.dim, u.segment, u.srch_id,
        sum(case when u.arm = 'T' then (u.num - s.treatment * u.den) / s.denom_t
                 else -(u.num - s.control * u.den) / s.denom_c end) as psi_gain,
        sum(case when u.arm = 'T' then (u.num - s.treatment * u.den) / (s.treatment * s.denom_t)
                 else -(u.num - s.control * u.den) / (s.control * s.denom_c) end) as psi_lift
    from ab_units u
    join s on s.dim = u.dim and s.segment = u.segment
    where u.test = 'top slots' and u.metric = 'ctr'
    group by u.dim, u.segment, u.srch_id
),
pair_se as (
    select dim, segment, sqrt(sum(d_gain * d_gain)) as se_gain, sqrt(sum(d_lift * d_lift)) as se_lift
    from (
        select
            p.dim, p.segment, x.srch_id,
            sum(case when x.segment = p.segment then x.psi_gain else -x.psi_gain end) as d_gain,
            sum(case when x.segment = p.segment then x.psi_lift else -x.psi_lift end) as d_lift
        from pairs p
        join psi x on x.dim = p.dim and x.segment in (p.segment, p.reference)
        group by p.dim, p.segment, x.srch_id
    )
    group by dim, segment
),
vs_ref as (
    select
        p.dim, p.segment, p.reference,
        p_two_sided((a.log_lift - r.log_lift) / e.se_lift)     as p_lift,
        p_two_sided((a.diff - r.diff) / e.se_gain)              as p_gain
    from pairs p
    join s a on a.dim = p.dim and a.segment = p.segment
    join s r on r.dim = p.dim and r.segment = p.reference
    join pair_se e on e.dim = p.dim and e.segment = p.segment
),
k as (
    select
        *,
        row_number() over (order by p_lift)                     as k_lift,
        row_number() over (order by p_gain)                     as k_gain,
        count(*) over ()                                        as m
    from vs_ref
),
bh as (
    select
        dim, segment, reference, p_lift, p_gain,
        least(1, min(p_lift * m / k_lift) over (
            order by k_lift desc rows between unbounded preceding and current row)) as p_bh_lift,
        least(1, min(p_gain * m / k_gain) over (
            order by k_gain desc rows between unbounded preceding and current row)) as p_bh_gain
    from k
)
select
    s.dim,
    s.segment,
    s.searches,
    s.denom_t + s.denom_c                                       as impressions,
    round(100 * s.treatment, 2)                                 as ctr_top_pct,
    round(100 * s.control, 2)                                   as ctr_next_pct,
    round(100 * (exp(s.log_lift) - 1), 1)                       as lift_pct,
    round(100 * (exp(s.log_lift - 1.96 * s.se_log_ratio) - 1), 1) as lift_lo,
    round(100 * (exp(s.log_lift + 1.96 * s.se_log_ratio) - 1), 1) as lift_hi,
    round(100 * s.diff, 2)                                      as gain_pts,
    round(100 * (s.diff - 1.96 * s.se_diff), 2)                 as gain_lo,
    round(100 * (s.diff + 1.96 * s.se_diff), 2)                 as gain_hi,
    b.reference                                                 as compared_with,
    round(b.p_bh_lift, 4)                                       as p_bh_lift,
    round(b.p_bh_gain, 4)                                       as p_bh_gain
from s
left join bh b on b.dim = s.dim and b.segment = s.segment
order by s.dim, s.level_no;
""")

In [ ]:
run("""
-- 4. Adjacent slots: is slot k really better than the next one down?
-- Each pair is compared within the searches that show both slots.
-- 32 tests, same BH treatment. Lower down the page the gaps get small enough
-- that most don't survive the correction.
create or replace view ab_adjacent as
with a as (
    select
        segment                                                 as comparison,
        cast(split_part(segment, ' ', 2) as int)                as upper_slot,
        searches,
        treatment,
        control,
        rel_lift, rel_lift_lo, rel_lift_hi,
        p_value
    from ab_results
    where test = 'adjacent slots'
),
r as (
    select *, row_number() over (order by p_value) as k, count(*) over () as m
    from a
)
select
    comparison,
    searches,
    round(100 * treatment, 2)                                   as ctr_upper_pct,
    round(100 * control, 2)                                     as ctr_lower_pct,
    round(100 * rel_lift, 1)                                    as lift_pct,
    round(100 * rel_lift_lo, 1)                                 as lift_lo,
    round(100 * rel_lift_hi, 1)                                 as lift_hi,
    round(p_value, 5)                                           as p_raw,
    round(least(1, min(p_value * m / k) over (order by k desc rows between unbounded preceding and current row)), 5) as p_bh,
    least(1, min(p_value * m / k) over (order by k desc rows between unbounded preceding and current row)) < 0.05 as significant_after_bh
from r
order by upper_slot;
""")

In [ ]:
run("""
-- 5. CUPED (variance reduction with a pre-experiment covariate).
-- Same test, same Feb-Jun searches, with and without the adjustment.
-- Variance reduction of v% is worth the same as running the test
-- 1 / (1 - v) times as long.
-- Result: it cuts variance by 1.5%. A single click is too noisy for hotel
-- traits to predict: the correlation between click and covariate is only
-- 0.09 (squared, under 1%). CUPED pays off on user-level
-- metrics with a strong history, like revenue per user, not on one click.
-- The adjustment does move the estimate down (4.08 to 3.86 points, +76% to
-- +71%), in line with it correcting for the location score 2 imbalance found
-- in 02 (location score 2 band is part of the covariate).
create or replace view ab_cuped as
select
    segment                                                     as version,
    searches,
    round(100 * treatment, 3)                                   as ctr_top_pct,
    round(100 * control, 3)                                     as ctr_next_pct,
    round(100 * diff, 3)                                        as diff_pts,
    round(100 * se_diff, 4)                                     as se_pts,
    round(100 * (1 - pow(se_diff / first_value(se_diff) over (order by segment desc), 2)), 1) as variance_cut_pct
from ab_results
where test = 'top slots, Feb-Jun'
order by segment desc;
""")

In [ ]:
run("""
-- 6. Cross-check for the imbalance found in 02: compare each hotel only with
-- itself. For hotels shown in both blocks across different random searches,
-- take CTR in slots 1-4 minus CTR in slots 6-10, and average with weights
-- n_t * n_c / (n_t + n_c), the usual precision weight for a difference.
-- Hotel quality is held fixed by construction.
-- No interval here: the weights depend on the data and many hotels appear
-- only once in a block, so closed-form errors (binomial, or clustered by
-- search) come out too narrow. The point estimate is what this check is for.
create or replace view ab_same_hotel as
with h as (
    select
        hotel_id,
        count(*) filter (where slot_block = '1-4')              as n_t,
        count(*) filter (where slot_block = '6-10')             as n_c,
        avg(case when clicked then 1.0 else 0 end) filter (where slot_block = '1-4')  as ctr_t,
        avg(case when clicked then 1.0 else 0 end) filter (where slot_block = '6-10') as ctr_c
    from impressions
    where is_random and slot_block in ('1-4', '6-10') and not reserved_slot
      and srch_id in (select srch_id from searches where full_top10)
    group by hotel_id
),
w as (
    select *, n_t * n_c * 1.0 / (n_t + n_c) as wt
    from h
    where n_t > 0 and n_c > 0
)
select
    count(*)                                                    as hotels,
    sum(n_t + n_c)                                              as impressions,
    round(100 * sum(wt * (ctr_t - ctr_c)) / sum(wt), 2)         as diff_pts,
    round(100 * sum(wt * (ctr_t - ctr_c)) / sum(wt * ctr_c), 1) as lift_pct
from w;


-- 7. Why the full-list filter: the same click-through comparison on every
-- random search (the first version of this test) and on the full-list
-- searches the test uses. Point estimates only; the second row is the same
-- comparison as ab_headline.
create or replace view ab_list_length as
with b as (
    select
        i.slot_block = '1-4'                                    as top,
        case when i.clicked then 1.0 else 0 end                 as click,
        s.full_top10
    from impressions i
    join searches s on s.srch_id = i.srch_id
    where i.is_random and not i.reserved_slot and i.slot_block in ('1-4', '6-10')
),
v as (
    select 1 as k, 'every random search' as searches_used,
           avg(click) filter (where top) as ctr_t, avg(click) filter (where not top) as ctr_c
    from b
    union all
    select 2, 'full-list searches only (full_top10)',
           avg(click) filter (where top), avg(click) filter (where not top)
    from b
    where full_top10
)
select
    searches_used,
    round(100 * ctr_t, 2)                                       as ctr_top_pct,
    round(100 * ctr_c, 2)                                       as ctr_next_pct,
    round(100 * (ctr_t / ctr_c - 1), 1)                         as lift_pct
from v
order by k;
""")

In [ ]:
q("select * from ab_headline")

In [ ]:
d = q("select * from ab_segments")
d = d.iloc[::-1].reset_index(drop=True)
overall = q("select rel_lift, diff from ab_results where test = 'top slots' and metric = 'ctr' and dim = 'all'").iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
y = np.arange(len(d))
ax[0].errorbar(d.lift_pct, y, xerr=[d.lift_pct - d.lift_lo, d.lift_hi - d.lift_pct], fmt="o", color=BLUE, capsize=2)
ax[0].axvline(100 * overall.rel_lift, color=GREY, ls="--", lw=1)
ax[0].set_xlabel("relative lift in click-through, %")
ax[1].errorbar(d.gain_pts, y, xerr=[d.gain_pts - d.gain_lo, d.gain_hi - d.gain_pts], fmt="o", color=RED, capsize=2)
ax[1].axvline(100 * overall["diff"], color=GREY, ls="--", lw=1)
ax[1].set_xlabel("absolute gain, clicks per 100 impressions")
ax[0].set_yticks(y); ax[0].set_yticklabels(d.dim + ": " + d.segment, fontsize=8)
fig.suptitle("Slots 1-4 vs 6-10 by segment: the relative lift varies a lot, the absolute gain much less", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
q("select * from ab_segments")

In [ ]:
q("select * from ab_adjacent")

In [ ]:
q("select * from ab_same_hotel")

In [ ]:
q("select * from ab_list_length")

In [ ]:
q("select * from ab_cuped")

In [ ]:
q("select * from ab_cuped_covariate")

## 5. Expedia's ranking vs random order

Booked searches only, compared within site, then averaged with the same site weights for both arms.

In [ ]:
run("""
-- 05_ranked_vs_random.sql
-- The comparison everyone wants first: Expedia's ranking against random order.
--
-- The naive version (booking rate per search) is in rank_naive and it's
-- meaningless here, see 02: the two groups were sampled differently.
--
-- What can be compared: searches that ended in a booking. Assuming Expedia
-- kept booked searches at a similar rate in both groups, the booked searches
-- in each group are a fair sample of booked searches under that ordering.
-- Caveats:
--   * conditioning on "booked" is conditioning on an outcome. If the ranking
--     changes WHO books, the groups differ for that reason too.
--   * the random share differs by site (02), so every comparison here is done
--     within site and then averaged with the same site weights for both arms
--     (direct standardisation).
""")

In [ ]:
run("""
-- 1. The naive readout, kept so the mistake is visible
create or replace view rank_naive as
select
    case when is_random then 'random order' else 'Expedia ranking' end as arm,
    count(*)                                                    as searches,
    round(100.0 * avg(case when has_booking then 1 else 0 end), 1) as booking_rate_pct,
    'not comparable: sampling differs by arm'                   as note
from searches
group by 1
order by 1;
""")

In [ ]:
run("""
-- 2. Booked searches, compared within site
create or replace table rank_booked as
select
    s.srch_id,
    s.site_id,
    s.is_random,
    s.booked_position,
    case when s.booked_position <= 4 then 1.0 else 0 end        as booked_in_top4,
    s.clicks,
    s.deepest_click_position,
    s.booking_value,
    i.price_pct_in_search                                       as booked_price_pct,
    i.stars                                                     as booked_stars,
    i.review_score                                              as booked_review
from searches s
join impressions i on i.srch_id = s.srch_id and i.booked
where s.has_booking;


create or replace view rank_booked_compare as
with long as (
    select site_id, is_random, 'position of booked hotel' as metric, booked_position * 1.0 as v from rank_booked
    union all select site_id, is_random, 'share booked from slots 1-4', booked_in_top4 from rank_booked
    union all select site_id, is_random, 'hotels clicked', clicks * 1.0 from rank_booked
    union all select site_id, is_random, 'deepest click position', deepest_click_position * 1.0 from rank_booked
    union all select site_id, is_random, 'booked hotel price percentile in search', booked_price_pct from rank_booked
    union all select site_id, is_random, 'booked hotel stars', booked_stars * 1.0 from rank_booked where booked_stars is not null
    union all select site_id, is_random, 'booked hotel review score', booked_review from rank_booked where booked_review is not null
    union all select site_id, is_random, 'booking value (USD)', booking_value from rank_booked where booking_value > 0
),
by_site as (
    select
        metric, site_id,
        avg(v) filter (where not is_random)                     as m_ranked,
        avg(v) filter (where is_random)                         as m_random,
        var_samp(v) filter (where not is_random)                as v_ranked,
        var_samp(v) filter (where is_random)                    as v_random,
        count(*) filter (where not is_random)                   as n_ranked,
        count(*) filter (where is_random)                       as n_random,
        count(*)                                                as n
    from long
    group by metric, site_id
    -- a site needs both arms (and a few of each) to be compared within
    having count(*) filter (where is_random) >= 20 and count(*) filter (where not is_random) >= 20
),
w as (
    select *, n * 1.0 / sum(n) over (partition by metric) as wt from by_site
),
raw as (
    select
        metric,
        avg(v) filter (where not is_random)                     as raw_ranked,
        avg(v) filter (where is_random)                         as raw_random
    from long
    group by metric
)
select
    w.metric,
    count(*)                                                    as sites_used,
    sum(w.n_ranked)                                             as booked_ranked,
    sum(w.n_random)                                             as booked_random,
    round(max(r.raw_ranked), 3)                                 as raw_ranked,
    round(max(r.raw_random), 3)                                 as raw_random,
    round(sum(w.wt * w.m_ranked), 3)                            as std_ranked,
    round(sum(w.wt * w.m_random), 3)                            as std_random,
    round(sum(w.wt * (w.m_ranked - w.m_random)), 3)             as diff,
    round(1.96 * sqrt(sum(w.wt * w.wt * (w.v_ranked / w.n_ranked + w.v_random / w.n_random))), 3) as ci95,
    p_two_sided(sum(w.wt * (w.m_ranked - w.m_random))
                / sqrt(sum(w.wt * w.wt * (w.v_ranked / w.n_ranked + w.v_random / w.n_random)))) as p_value
from w
join raw r on r.metric = w.metric
group by w.metric
order by case w.metric
    when 'position of booked hotel' then 1 when 'share booked from slots 1-4' then 2
    when 'hotels clicked' then 3 when 'deepest click position' then 4
    when 'booked hotel price percentile in search' then 5 when 'booked hotel stars' then 6
    when 'booked hotel review score' then 7 else 8 end;
""")

In [ ]:
q("select * from rank_naive")

In [ ]:
q("select * from rank_booked_compare")

## 6. Power and sample size, from the variance this traffic actually has

Two designs: within-search (the slot test, each search compared with itself) and between-search (whole searches split 50/50, the usual product test).

In [ ]:
run("""
-- 06_power.sql
-- What a test on this traffic can detect, worked out from the variance
-- actually observed, not from a textbook p(1-p) guess.
--
-- Two designs:
--
--   within-search (pow_mde, pow_sample_size): the slot test in 04. Every
--   search has hotels in both arms (some in 1-4, some in 6-10), so each
--   search is compared with itself.
--
--   between-search (pow_between): the usual product test, where whole
--   searches are split 50/50, say old ranking vs new ranking. Each search
--   sits in one arm.
--
-- They use different metrics (click-through on nine top slots vs on the whole
-- list), so compare metrics within a design, not across designs.
--
-- From ab_results: se = sigma / sqrt(n searches), so sigma = se * sqrt(n).
-- Minimum detectable effect at 80% power and 5% two-sided:
--   within-search    MDE = (1.96 + 0.84) * sigma / sqrt(n)
--   between-search   MDE = (1.96 + 0.84) * 2 * sigma / sqrt(n)    (n/2 per arm)
-- and the searches needed for a relative lift L on a control rate c:
--   n = ((1.96 + 0.84) * sigma / (L * c))^2, times 4 for between-search.
--
-- Traffic = the random-order searches in this file, a sample of Expedia's
-- real traffic, per 7 days over the days the file covers. The slot test only uses searches with all nine top slots
-- (1-4 and 6-10, 88% of them), and "booked after click" only exists in
-- searches with a click in those slots (52%), so traffic is scaled by those
-- shares. The weeks are illustrative; the ratios between metrics and between
-- designs are the useful part.


create or replace view pow_traffic as
select
    count(*)                                                    as random_searches,
    datediff('day', min(search_date), max(search_date)) + 1     as days,
    round((datediff('day', min(search_date), max(search_date)) + 1) / 7.0, 1) as weeks,
    round(count(*) * 7.0 / (datediff('day', min(search_date), max(search_date)) + 1)) as random_searches_per_week
from searches
where is_random;


create or replace view pow_mde as
with s as (
    select
        metric,
        control,
        se_diff * sqrt(searches)                                as sigma,
        searches * 1.0 / (select random_searches from pow_traffic) as share_of_searches,
        se_ratio_vs_naive
    from ab_results
    where test = 'top slots' and dim = 'all'
),
t as (select random_searches * 7.0 / days as per_week from pow_traffic),
weeks as (select unnest([1, 2, 4, 8, 16]) as w)
select
    s.metric,
    w.w                                                         as weeks,
    cast(w.w * t.per_week * s.share_of_searches as int)         as searches,
    round(100 * 2.8 * s.sigma / sqrt(w.w * t.per_week * s.share_of_searches) / s.control, 1) as mde_relative_pct,
    round(s.se_ratio_vs_naive, 2)                               as se_vs_naive
from s
cross join t
cross join weeks w
order by s.metric, w.w;


-- how many random-order searches it takes to detect a given relative lift
create or replace view pow_sample_size as
with s as (
    select
        metric, control, se_diff * sqrt(searches) as sigma,
        searches * 1.0 / (select random_searches from pow_traffic) as share_of_searches
    from ab_results
    where test = 'top slots' and dim = 'all'
),
t as (select random_searches * 7.0 / days as per_week from pow_traffic),
lifts as (select unnest([0.02, 0.05, 0.10, 0.20]) as lift)
select
    s.metric,
    round(100 * l.lift)                                         as lift_pct,
    cast(ceil(pow(2.8 * s.sigma / (l.lift * s.control), 2)) as bigint) as searches_needed,
    round(pow(2.8 * s.sigma / (l.lift * s.control), 2) / (t.per_week * s.share_of_searches), 1) as weeks_of_random_traffic
from s
cross join lifts l
cross join t
order by s.metric, l.lift;


-- Between-search design. Metrics per search: click-through (clicks / hotels
-- shown, a ratio, so sigma comes from the same delta-method linearisation as
-- 04), booking rate and booking value per search.
-- Caveat: every search in this file has at least one click (02). Real traffic
-- has many searches with none, so click-through varies more there and the
-- CTR row below is optimistic. The booking rows don't have that problem to
-- the same degree, but their base rate is set by the sampling too.
create or replace view pow_between as
with s as (
    select
        clicks * 1.0                                            as clicks,
        hotels_shown * 1.0                                      as shown,
        case when has_booking then 1.0 else 0 end               as booked,
        booking_value
    from searches
    where is_random
),
m as (
    select sum(clicks) / sum(shown) as ctr, avg(shown) as shown_bar from s
),
sig as (
    select 'ctr' as metric, any_value(m.ctr) as mean,
           stddev_samp((s.clicks - m.ctr * s.shown) / m.shown_bar) as sigma
    from s cross join m
    union all
    select 'booking rate per search', avg(booked), stddev_samp(booked) from s
    union all
    select 'booking value per search', avg(booking_value), stddev_samp(booking_value) from s
),
t as (select random_searches * 7.0 / days as per_week from pow_traffic),
weeks as (select unnest([1, 2, 4, 8, 16]) as w)
select
    sig.metric,
    w.w                                                         as weeks,
    cast(w.w * t.per_week as int)                               as searches,
    round(100 * 2.8 * 2 * sig.sigma / sqrt(w.w * t.per_week) / sig.mean, 1) as mde_relative_pct,
    round(pow(2.8 * 2 * sig.sigma / (0.05 * sig.mean), 2) / t.per_week, 1)   as weeks_for_5pct_lift
from sig
cross join t
cross join weeks w
order by sig.metric, w.w;
""")

In [ ]:
q("select * from pow_traffic")

In [ ]:
q("select * from pow_mde")

In [ ]:
q("select * from pow_sample_size")

In [ ]:
q("select * from pow_between where weeks = 4")

## 7. A/A tests and peeking

In an A/A test both groups get exactly the same thing, so any "significant" difference is a false positive and over many random splits the share of p < 0.05 should be 5%. If it isn't, the test is broken.

Checked here:

1. Unit of analysis. Splitting whole searches (the between-search design in section 6) but computing the error as if every impression were independent: does that break the 5%?
2. Peeking. Checking the result every week and stopping the first time p < 0.05: how far above 5% does that push false positives, and does an O'Brien-Fleming boundary fix it?

Metric: click-through rate (clicks / impressions), randomised by search, standard error by the delta method with searches as the unit.

In [ ]:
import pathlib

import duckdb
import numpy as np
from scipy.stats import norm

ROOT = WORK
OUT = ROOT / "results"
RNG = np.random.default_rng(2013)
N_SIMS = 5000

con = db
df = con.sql("""
    select
        s.srch_id,
        datediff('week', date '2012-10-29', week_start(s.search_date)) as week_no,
        s.clicks                                         as y,
        s.hotels_shown                                   as x,
        case when s.has_booking then 1 else 0 end        as booked
    from searches s
    where s.is_random
    order by s.search_ts, s.srch_id
""").df()

y = df["y"].to_numpy(float)
x = df["x"].to_numpy(float)
week = df["week_no"].to_numpy()
n_weeks = week.max() + 1


def ratio_test(y, x, g):
    """CTR difference between g==1 and g==0, delta-method se with searches as units."""
    out = {}
    for arm in (0, 1):
        m = g == arm
        r = y[m].sum() / x[m].sum()
        xbar = x[m].mean()
        lin = (y[m] - r * x[m]) / xbar
        out[arm] = (r, lin.var(ddof=1) / m.sum(), x[m].sum())
    diff = out[1][0] - out[0][0]
    se = np.sqrt(out[0][1] + out[1][1])
    naive_se = np.sqrt(sum(r * (1 - r) / n for r, _, n in out.values()))
    return diff / se, diff / naive_se


# ---- 1. plain A/A: N_SIMS random 50/50 splits of searches ----
z_delta = np.empty(N_SIMS)
z_naive = np.empty(N_SIMS)
for i in range(N_SIMS):
    g = RNG.integers(0, 2, len(y))
    z_delta[i], z_naive[i] = ratio_test(y, x, g)

p_delta = 2 * norm.sf(np.abs(z_delta))
p_naive = 2 * norm.sf(np.abs(z_naive))

aa = [
    ("delta method (search as unit)", (p_delta < 0.05).mean()),
    ("naive (impression as unit)", (p_naive < 0.05).mean()),
]
# a false-positive rate estimated from N_SIMS runs has its own noise
fp_se = np.sqrt(0.05 * 0.95 / N_SIMS)
# z_delta / z_naive = naive se / delta-method se, for the same split
se_vs_delta = {"delta method (search as unit)": 1.0,
               "naive (impression as unit)": float(np.median(z_delta / z_naive))}
with open(OUT / "aa_false_positive_rate.csv", "w") as f:
    f.write("method,simulations,false_positive_rate_pct,expected_pct,expected_band_95,median_se_vs_delta_method\n")
    for name, rate in aa:
        f.write(f"{name},{N_SIMS},{100 * rate:.2f},5.00,"
                f"{100 * (0.05 - 1.96 * fp_se):.2f} to {100 * (0.05 + 1.96 * fp_se):.2f},"
                f"{se_vs_delta[name]:.2f}\n")

# p-value histogram for the chart: under a working test it's flat
bins = np.linspace(0, 1, 21)
with open(OUT / "aa_pvalue_histogram.csv", "w") as f:
    f.write("bin_start,bin_end,delta_method,naive\n")
    hd, _ = np.histogram(p_delta, bins)
    hn, _ = np.histogram(p_naive, bins)
    for a, b, c, d in zip(bins[:-1], bins[1:], hd, hn):
        f.write(f"{a:.2f},{b:.2f},{c},{d}\n")


# ---- 2. peeking ----
# Weekly looks. For each simulated A/A split, compute the z-score on all data
# up to week k, for every k. Running sums make that cheap:
#   sum((y - r x)^2) = sum(y^2) - 2 r sum(xy) + r^2 sum(x^2)
def weekly_z(g):
    z = np.empty(n_weeks)
    stats = {}
    for arm in (0, 1):
        m = g == arm
        w = week[m]
        cum = [np.cumsum(np.bincount(w, weights=v, minlength=n_weeks))
               for v in (y[m], x[m], y[m] ** 2, x[m] ** 2, x[m] * y[m], np.ones(m.sum()))]
        stats[arm] = cum
    for k in range(n_weeks):
        parts = []
        for arm in (0, 1):
            sy, sx, syy, sxx, sxy, n = (c[k] for c in stats[arm])
            r = sy / sx
            xbar = sx / n
            ss = syy - 2 * r * sxy + r * r * sxx
            parts.append((r, ss / (n - 1) / xbar ** 2 / n))
        z[k] = (parts[1][0] - parts[0][0]) / np.sqrt(parts[0][1] + parts[1][1])
    return z


# O'Brien-Fleming style boundary: reject at look k if |z_k| > c / sqrt(t_k),
# where t_k is the share of the final sample seen so far (the "information
# fraction"). Weeks aren't equal in size here (traffic grows from November to
# June), so t_k uses the actual cumulative search counts, not k / K.
# c is calibrated by simulating the null as Brownian motion observed at those
# t_k, so the overall false-positive rate is 5%.
def obf_constant(t, alpha=0.05, sims=200_000):
    dt = np.diff(np.concatenate([[0.0], t]))
    paths = np.cumsum(RNG.standard_normal((sims, len(t))) * np.sqrt(dt), axis=1)
    z = paths / np.sqrt(t)
    return np.quantile((np.abs(z) * np.sqrt(t)).max(axis=1), 1 - alpha)


# weeks are numbered from 0, and week 0 is only 4 days (1-4 Nov 2012). The
# first look is at week index 2, after 2 weeks and 4 days of data, so it
# isn't run on a handful of searches
first_look = 2
K = n_weeks - first_look
cum_n = np.cumsum(np.bincount(week, minlength=n_weeks))[first_look:]
info = cum_n / cum_n[-1]
c_obf = obf_constant(info)

stop_naive = np.zeros(N_SIMS, bool)
stop_obf = np.zeros(N_SIMS, bool)
final_only = np.zeros(N_SIMS, bool)
first_stop_week = []
for i in range(N_SIMS):
    g = RNG.integers(0, 2, len(y))
    z = weekly_z(g)[first_look:]
    hits = np.abs(z) > 1.96
    stop_naive[i] = hits.any()
    if hits.any():
        first_stop_week.append(int(np.argmax(hits)) + first_look + 1)
    stop_obf[i] = (np.abs(z) > c_obf / np.sqrt(info)).any()
    final_only[i] = abs(z[-1]) > 1.96

with open(OUT / "peeking.csv", "w") as f:
    f.write("rule,looks,false_positive_rate_pct\n")
    f.write(f"one look at the end,1,{100 * final_only.mean():.1f}\n")
    f.write(f"look every week; stop at first p < 0.05,{K},{100 * stop_naive.mean():.1f}\n")
    f.write(f"look every week; O'Brien-Fleming boundary (c = {c_obf:.2f}),{K},{100 * stop_obf.mean():.1f}\n")

# the false "wins" show up mostly early, when the data is thin
with open(OUT / "peeking_first_false_win.csv", "w") as f:
    f.write("week,false_wins\n")
    counts = np.bincount(first_stop_week, minlength=n_weeks + 1)
    for wk in range(first_look + 1, n_weeks + 1):
        f.write(f"{wk},{counts[wk]}\n")

print(open(OUT / "aa_false_positive_rate.csv").read())
print(open(OUT / "peeking.csv").read())

In [ ]:
h = pd.read_csv(WORK / "results" / "aa_pvalue_histogram.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for a, colname, title in [(ax[0], "delta_method", "search as unit: flat, as it should be"),
                          (ax[1], "naive", "impression as unit: skewed toward 1")]:
    a.bar(h.bin_start, h[colname], width=0.05, align="edge", color=BLUE if colname == "delta_method" else RED)
    a.set_title(title, fontsize=10); a.set_xlabel("p-value")
fig.suptitle("A/A p-values over 5,000 random splits", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
pd.read_csv(WORK / "results" / "peeking.csv")

## 8. Geo-lift test design

The question a marketing team brings: "we want to run a campaign in a few countries and measure the lift. Which countries, for how long, and what's the smallest lift we'd be able to see?"

No campaign ran in this data, so the lift is added by hand: multiply a market's real weekly searches by (1 + lift) over a test window, then check whether synthetic control finds it. Doing that over many windows and lift sizes gives the power curve. This is how GeoLift-style tools size a test before it runs.

Synthetic control builds the treated market's "no campaign" path as a weighted mix of the other markets, with weights >= 0 that sum to 1, fitted on the weeks before the test. The lift is actual vs that mix during the test.

Inference: placebo tests. Pretend each untreated market (or group of markets) was the treated one, run the same estimate, and see how big the fake lifts get. Those fake lifts are the null distribution: a real lift has to land above their 97.5th percentile (a two-sided 5% test, equal tails). The fake lifts aren't centred on zero here (spring 2013 shifted some markets against others), so equal tails are used rather than |lift|.

Placebos only come from mid-size markets (100+ searches a week). My first version used every market, and the small ones (31-85 searches a week, about 1.5 times as noisy week to week) widened the critical value: a single market couldn't detect even a 40% lift, and three markets needed 25% instead of 15% over 6 weeks (the last table in this section). A placebo should be about as noisy as the market being tested. The cost is a small pool: 10 mid-size markets, so 35 placebo groups per test window for a three-market design and 9 for a single market.

Market choice uses only weeks 0-17, before the earliest test window: fit on weeks 0-11, score the fit on weeks 12-17. Choosing markets on the same weeks the power curve is run on would make the design look better than it is.

In [ ]:
run("""
-- 07_geo_panel.sql
-- Panel for the geo-lift work in stats/geo_lift.py: weekly searches per
-- visitor country.
--
-- There is no real geo campaign in this data. The point of the geo part is
-- the design: which markets to test in, how long, what lift is detectable,
-- and checking that the method finds a lift when one is there. That is done
-- by adding a known lift to real market series and seeing if synthetic
-- control recovers it (the same way Meta's GeoLift runs power analysis).
--
-- Weeks run Monday to Sunday, 5 Nov 2012 to 30 Jun 2013 (34 full weeks).
-- Countries need 30+ searches a week on average to be usable. Country ids
-- are anonymised in the source; 219 is by far the largest.


create or replace view geo_weekly as
with w as (
    select
        visitor_country                                         as country,
        week_start(search_date)                                 as week,
        count(*)                                                as searches,
        count(*) filter (where has_booking)                     as bookings
    from searches
    where search_date between date '2012-11-05' and date '2013-06-30'
    group by 1, 2
),
keep as (
    select country
    from w
    group by country
    having sum(searches) >= 30 * 34 and count(*) = 34
)
select w.*
from w
join keep k using (country)
order by country, week;


-- one row per market: size and how noisy it is week to week
create or replace view geo_markets as
with s as (
    select
        country,
        avg(searches)                                           as avg_weekly_searches,
        stddev_samp(searches) / avg(searches)                   as cv
    from geo_weekly
    group by country
)
select
    country,
    round(avg_weekly_searches)                                  as avg_weekly_searches,
    round(cv, 3)                                                as week_to_week_cv,
    rank() over (order by avg_weekly_searches desc)             as size_rank
from s
order by avg_weekly_searches desc;


-- How noisy small markets are next to the mid-size ones the placebos come
-- from (mid-size = 100+ searches a week, not counting the largest market).
create or replace view geo_noise_by_size as
with s as (
    select
        country,
        avg(searches)                                           as avg_weekly,
        stddev_samp(searches) / avg(searches)                   as cv
    from geo_weekly
    group by country
)
select
    case when country = 219 then 'largest market (219)'
         when avg_weekly >= 100 then 'mid-size (100+ a week)'
         else 'small (under 100 a week)' end                    as market_size,
    count(*)                                                    as markets,
    round(min(avg_weekly))                                      as min_avg_weekly_searches,
    round(max(avg_weekly))                                      as max_avg_weekly_searches,
    round(avg(cv), 3)                                           as mean_week_to_week_cv
from s
group by 1
order by min(avg_weekly) desc;
""")

In [ ]:
q("select * from geo_markets")

In [ ]:
q("select * from geo_noise_by_size")

In [ ]:
import itertools
import pathlib

import duckdb
import numpy as np
from scipy.optimize import nnls

ROOT = WORK
OUT = ROOT / "results"
RNG = np.random.default_rng(7)

con = db
panel = con.sql("select country, week, searches from geo_weekly order by country, week").df()
wide = panel.pivot(index="week", columns="country", values="searches")
weeks = wide.index.to_list()
markets = [int(c) for c in wide.columns]
Y = wide.to_numpy(float)                # weeks x markets
T_WEEKS = Y.shape[0]
LARGEST = 219                           # by far the largest market (geo_markets), a donor only

MIN_PRE = 18
LIFTS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
LENGTHS = [2, 4, 6, 8]


def col(m):
    return markets.index(m)


def sc_fit(y_pre, X_pre):
    """Weights >= 0 summing to 1, least squares on the pre-period.
    The sum-to-one constraint is added as an extra heavily weighted row,
    which keeps it a plain non-negative least squares problem."""
    lam = 1e4
    A = np.vstack([X_pre, lam * np.ones(X_pre.shape[1])])
    b = np.concatenate([y_pre, [lam]])
    w, _ = nnls(A, b)
    return w / w.sum()


def estimate(Ymat, treated, start, length, lift=0.0, donors=None):
    """Estimated relative lift for a treated group of markets.

    Each series is divided by its own pre-period mean so big and small
    markets are on the same scale. Returns lift, weights, and the actual and
    synthetic (normalised) series."""
    tcols = [col(m) for m in treated]
    if donors is None:
        donors = [m for m in markets if m not in treated]
    dcols = [col(m) for m in donors]
    y = Ymat[:, tcols].sum(axis=1).copy()
    y[start:start + length] *= 1 + lift
    pre = slice(0, start)
    y_n = y / y[pre].mean()
    X = Ymat[:, dcols]
    X_n = X / X[pre].mean(axis=0)
    w = sc_fit(y_n[pre], X_n[pre])
    synth = X_n @ w
    post = slice(start, start + length)
    est = y_n[post].sum() / synth[post].sum() - 1
    return est, w, y_n, synth


MID_SIZE = 100   # average weekly searches


def placebo_lifts(treated, start, length, n_draws=200, min_size=MID_SIZE, rng=RNG):
    """Fake lifts from pretending other markets (same group size) were treated.
    The real treated markets are left out of the donor pool for these.
    min_size=0 lets the small markets back in (the check in section 5)."""
    pool = [m for m in markets if m not in treated and m != LARGEST
            and Y[:, col(m)].mean() >= min_size]
    k = len(treated)
    combos = list(itertools.combinations(pool, k))
    if len(combos) > n_draws:
        idx = rng.choice(len(combos), n_draws, replace=False)
        combos = [combos[i] for i in idx]
    out = []
    for fake in combos:
        donors = [m for m in markets if m not in fake and m not in treated]
        est, *_ = estimate(Y, list(fake), start, length, donors=donors)
        out.append(est)
    return np.array(out)


# ---- 1. how well can each market be rebuilt from the others? ----
# fit on weeks 0-11, score on weeks 12-17. In-sample fit always looks good,
# so the ranking uses the out-of-sample error. Everything here is before
# MIN_PRE, the first week any test window can start.
SEL_FIT, SEL_END = 12, MIN_PRE
rows = []
for m in markets:
    est, w, y_n, synth = estimate(Y, [m], start=SEL_FIT, length=SEL_END - SEL_FIT)
    pre_err = np.sqrt(np.mean((y_n[:SEL_FIT] - synth[:SEL_FIT]) ** 2))
    post_err = np.sqrt(np.mean((y_n[SEL_FIT:SEL_END] - synth[SEL_FIT:SEL_END]) ** 2))
    top = sorted(zip(w, [d for d in markets if d != m]), reverse=True)[:3]
    rows.append((m, Y[:, col(m)].mean(), pre_err, post_err, est,
                 "; ".join(f"{d}:{wt:.2f}" for wt, d in top if wt > 0.005)))
rows.sort(key=lambda r: r[3])
with open(OUT / "geo_market_fit.csv", "w") as f:
    f.write("country,avg_weekly_searches,fit_error_pre_pct,fit_error_holdout_pct,holdout_fake_lift_pct,top_donor_weights\n")
    for m, size, pe, po, est, tw in rows:
        f.write(f"{m},{size:.0f},{100 * pe:.1f},{100 * po:.1f},{100 * est:.1f},{tw}\n")


# ---- 2. candidate designs ----
# single markets: the mid-size ones (not the giant, not the tiny)
mid = [m for m in markets if m != LARGEST and Y[:, col(m)].mean() >= MID_SIZE]
fit_rank = {r[0]: i for i, r in enumerate(rows)}
best_single = min(mid, key=lambda m: fit_rank[m])
# group of three: the three mid-size markets with the best holdout fit
group3 = sorted(mid, key=lambda m: fit_rank[m])[:3]
designs = {f"one market: {best_single}": [best_single],
           "three markets: " + " + ".join(map(str, group3)): group3}


# ---- 3. power curves ----
power_rows = []
for name, treated in designs.items():
    for L in LENGTHS:
        starts = list(range(MIN_PRE, T_WEEKS - L + 1))
        # critical value: 97.5th percentile of the fake lifts, pooled across
        # placebo groups and across all test windows of this length
        null = np.concatenate([placebo_lifts(treated, s, L) for s in starts])
        crit_lo, crit = np.quantile(null, [0.025, 0.975])
        for lift in LIFTS:
            ests = np.array([estimate(Y, treated, s, L, lift=lift)[0] for s in starts])
            power_rows.append((name, L, lift, len(starts), len(null) // len(starts), crit,
                               ests.mean(), ((ests < crit_lo) | (ests > crit)).mean(),
                               (ests > crit).mean()))

# power_pct: share of windows above the upper critical value (a lift found).
# flagged_either_tail_pct: share beyond either 2.5% tail, the two-sided test;
# at 0% lift that is the design's false-positive rate.
with open(OUT / "geo_power.csv", "w") as f:
    f.write("design,test_weeks,true_lift_pct,windows_tested,placebos_per_window,critical_lift_pct,"
            "mean_estimated_lift_pct,flagged_either_tail_pct,power_pct\n")
    for name, L, lift, n, ppw, crit, mean_est, either, pw in power_rows:
        f.write(f"{name},{L},{100 * lift:.1f},{n},{ppw},{100 * crit:.1f},{100 * mean_est:.1f},"
                f"{100 * either:.0f},{100 * pw:.0f}\n")

# smallest lift with at least 80% power, per design and length.
# With 9 to 15 windows per design, the power estimates are coarse; treat the
# MDE as a ballpark.
with open(OUT / "geo_mde.csv", "w") as f:
    f.write("design,test_weeks,mde_80pct_power\n")
    for name in designs:
        for L in LENGTHS:
            ok = [lift for (n, l, lift, *_r, pw) in power_rows if n == name and l == L and pw >= 0.8 and lift > 0]
            f.write(f"{name},{L},{'%.1f%%' % (100 * min(ok)) if ok else 'over %.0f%%' % (100 * max(LIFTS))}\n")


# ---- 4. two worked readouts ----
# simulated campaigns in the three-market group over the last 6 weeks
# (20 May to 30 June 2013): one at 10%, a lift a marketer would be happy with,
# and one at the size the power analysis says is detectable.
L = 6
start = T_WEEKS - L
g3_name = [n for n in designs if n.startswith("three")][0]
mde6 = [lift for (n, l, lift, *_r, pw) in power_rows if n == g3_name and l == L and pw >= 0.8 and lift > 0]
big_lift = min(mde6) if mde6 else 0.30
# pool placebo fake lifts across every 6-week window, not just the last one:
# a single window only has a few dozen placebo groups
null = np.concatenate([placebo_lifts(group3, s, L) for s in range(MIN_PRE, T_WEEKS - L + 1)])


def readout(true_lift):
    est, w, y_n, synth = estimate(Y, group3, start, L, lift=true_lift)
    # two-sided placebo p-value, equal tails
    upper = (np.sum(null >= est) + 1) / (len(null) + 1)
    lower = (np.sum(null <= est) + 1) / (len(null) + 1)
    p = min(1.0, 2 * min(upper, lower))
    return est, w, y_n, synth, p


# difference-in-differences against the raw total of all other markets, for
# comparison. A raw total weights markets by size, so 219 dominates it.
donors = [m for m in markets if m not in group3]
y_raw = Y[:, [col(m) for m in group3]].sum(axis=1).copy()
y_raw[start:] *= 1.10
d_raw = Y[:, [col(m) for m in donors]].sum(axis=1)
did = (y_raw[start:].mean() / y_raw[:start].mean()) / (d_raw[start:].mean() / d_raw[:start].mean()) - 1
share_largest = Y[:, col(LARGEST)].sum() / d_raw.sum()

with open(OUT / "geo_example_readout.csv", "w") as f:
    f.write("item,small_campaign,large_campaign\n")
    a = readout(0.10)
    b = readout(big_lift)
    f.write(f"treated markets,{' '.join(map(str, group3))},{' '.join(map(str, group3))}\n")
    f.write(f"test weeks,{weeks[start]:%Y-%m-%d} to {weeks[-1]:%Y-%m-%d},same\n")
    f.write(f"true lift added,10.0%,{100 * big_lift:.1f}%\n")
    f.write(f"synthetic control estimate,{100 * a[0]:.1f}%,{100 * b[0]:.1f}%\n")
    band = f"{100 * np.quantile(null, 0.025):.1f}% to {100 * np.quantile(null, 0.975):.1f}%"
    f.write(f"fake lifts with no campaign (95% range),{band},{band}\n")
    med = f"{100 * np.median(null):.1f}%"
    f.write(f"fake lifts with no campaign (median),{med},{med}\n")
    f.write(f"p-value (placebo rank),{a[4]:.3f},{b[4]:.3f}\n")
    f.write(f"placebo estimates,{len(null)},{len(null)}\n")
    f.write(f"difference-in-differences estimate,{100 * did:.1f}% (10% case),\n")
    f.write(f"country {LARGEST}'s share of the difference-in-differences comparison total,{100 * share_largest:.1f}%,same\n")
    f.write("donor weights," + "; ".join(f"{d}:{wt:.2f}" for wt, d in sorted(zip(a[1], donors), reverse=True) if wt > 0.005) + ",same\n")

with open(OUT / "geo_example_series.csv", "w") as f:
    f.write("week,actual_index_10pct,actual_index_large,synthetic_index,in_test\n")
    for i, wk in enumerate(weeks):
        f.write(f"{wk:%Y-%m-%d},{a[2][i]:.4f},{b[2][i]:.4f},{a[3][i]:.4f},{i >= start}\n")


# ---- 5. why placebos come only from mid-size markets ----
# The same power analysis with the small markets let back into the placebo
# pool (every market except the largest and the treated ones). Their noise
# widens the critical value. A separate random generator (pools over 200
# placebo groups are sampled) keeps every output above unchanged.
rng_check = np.random.default_rng(8)
with open(OUT / "geo_placebo_pool_check.csv", "w") as f:
    f.write("design,test_weeks,placebo_pool,placebos_per_window,critical_lift_pct,mde_80pct_power\n")
    for name, treated in designs.items():
        for L in LENGTHS:
            starts = list(range(MIN_PRE, T_WEEKS - L + 1))
            ests = {lift: np.array([estimate(Y, treated, s, L, lift=lift)[0] for s in starts])
                    for lift in LIFTS if lift > 0}
            for pool_name, min_size in [("mid-size markets only", MID_SIZE),
                                        (f"every market except {LARGEST}", 0)]:
                null = np.concatenate([placebo_lifts(treated, s, L, min_size=min_size, rng=rng_check)
                                       for s in starts])
                crit = np.quantile(null, 0.975)
                ok = [lift for lift, e in ests.items() if (e > crit).mean() >= 0.8]
                mde = "%.1f%%" % (100 * min(ok)) if ok else "over %.0f%%" % (100 * max(LIFTS))
                f.write(f"{name},{L},{pool_name},{len(null) // len(starts)},{100 * crit:.1f},{mde}\n")

print(open(OUT / "geo_market_fit.csv").read())
print(open(OUT / "geo_mde.csv").read())
print(open(OUT / "geo_example_readout.csv").read())

In [ ]:
p = pd.read_csv(WORK / "results" / "geo_power.csv")
fig, ax = plt.subplots()
for (design, weeks), g in p[p.test_weeks.isin([4, 6])].groupby(["design", "test_weeks"]):
    ax.plot(g.true_lift_pct, g.power_pct, marker="o",
            color=BLUE if design.startswith("three") else GREY, ls="-" if weeks == 6 else ":",
            label=f"{design}, {weeks} weeks")
ax.axhline(80, color=RED, lw=0.8, ls="--")
ax.set_xlabel("true lift added, %"); ax.set_ylabel("power, %")
ax.set_title("Detecting a geo campaign: three markets beat one")
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
s = pd.read_csv(WORK / "results" / "geo_example_series.csv", parse_dates=["week"])
fig, ax = plt.subplots()
ax.plot(s.week, s.actual_index_10pct, color=BLUE, marker="o", ms=3, label="treated markets (+10% in test weeks)")
ax.plot(s.week, s.synthetic_index, color=GREY, ls="--", label="synthetic control")
ax.axvspan(s.week[s.in_test].min(), s.week.max(), color=GREY, alpha=0.15)
ax.set_ylabel("weekly searches, index (pre-period mean = 1)")
ax.plot(s.week[s.in_test], s.actual_index_large[s.in_test], color=RED, marker="o", ms=3, alpha=0.6, label="treated markets (+15% in test weeks)")
ax.set_title("Worked example: simulated campaigns in three markets")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

In [ ]:
pd.read_csv(WORK / "results" / "geo_example_readout.csv")

In [ ]:
pd.read_csv(WORK / "results" / "geo_placebo_pool_check.csv")

## Limits

- The file is a sample selected on clicks and bookings, so absolute rates aren't Expedia's real ones. Differences inside the random group are what's trustworthy.
- A slot effect is measured with everything else shuffled. In a real ranking, moving one hotel up pushes another down.
- Conversion after a click compares people who chose to click, so it isn't randomised the way clicks and bookings per impression are.
- The geo-lift part adds simulated lifts to real market data. It's a design exercise, the power estimates rest on 9 to 15 test windows each, and the placebo pool is small (10 mid-size markets).